# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys,math, random, collections, csv,base64,io
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:

TASK_ID='task009'
ROOT=Path(COMPETITION)
if not (ROOT/f'{TASK_ID}.json').exists():
    # Local fallback for notebook verification outside Kaggle.
    ROOT=Path('/mnt/data')
TASK_PATH=ROOT/f'{TASK_ID}.json'

OUT_DIR=Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
HEALTH_PATH=OUT_DIR/f'{TASK_ID}_verified_onnx_health.json'

In [6]:
torch.set_num_threads(1)

CH=10; H=W=30


In [7]:
def onnx_shape(v):
    return [d.dim_value for d in v.type.tensor_type.shape.dim]

def op_counts(m):
    out={}
    for n in m.graph.node:
        out[n.op_type]=out.get(n.op_type,0)+1
    return out

def encode_grid(grid):
    arr=np.array(grid, dtype=np.int64)
    x=np.zeros((1,10,30,30), dtype=np.float32)
    h,w=arr.shape
    for c in range(10):
        x[0,c,:h,:w]=(arr==c).astype(np.float32)
    return x

def decode_grid(y,h,w):
    if y.ndim==4:
        y=y[0]
    return y[:,:h,:w].argmax(0).astype(np.int64)

def exact_eval(sess, cases):
    ok=0; first_wrong=None
    for i,ex in enumerate(cases):
        if 'output' not in ex:
            continue
        h=len(ex['input']); w=len(ex['input'][0])
        y=sess.run(None, {'input': encode_grid(ex['input'])})[0]
        pred=decode_grid(y,h,w)
        gold=np.array(ex['output'], dtype=np.int64)
        good=np.array_equal(pred,gold)
        ok += int(good)
        if (not good) and first_wrong is None:
            first_wrong={'index':i,'pred':pred.tolist(),'gold':gold.tolist()}
    return {'exact':ok,'total':len(cases),'first_wrong':first_wrong}


In [8]:

def _left(n):
    return torch.tensor(np.triu(np.ones((n,n),np.float32),1))

def _right(n):
    return torch.tensor(np.tril(np.ones((n,n),np.float32),-1))

def _rowmap(spacing, offset, n):
    cell=spacing-1
    m=np.zeros((30,n),np.float32)
    for k in range(n):
        for d in range(cell):
            p=offset+k*spacing+d
            if 0 <= p < 30:
                m[p,k]=1.0
    return torch.tensor(m)

class Task009MultiLatticeRelation(nn.Module):
    """Static ONNX relation-completion model for task009.

    The model is intentionally not an input-output bank. It generalizes the earlier
    fixed-lattice solution by considering several possible lattice offsets/spacings.
    For each lattice hypothesis it:
      1. detects solid non-background cell blocks;
      2. fills cells between same-color blocks sharing a lattice row or column;
      3. projects completed lattice cells back to pixels;
      4. unions the safe completions over all hypotheses.

    This removes Shape/Tile/repeat-interleave from the earlier task009 model and
    keeps the ONNX graph static and small.
    """
    def __init__(self):
        super().__init__()
        self.configs=[]
        # s=3 covers the visible generator; s=4 and s=5 are included for hidden robustness.
        # s=2 is excluded because 1x1 hypotheses can create many false positives on 2x2 blocks.
        for s in [3,4,5]:
            for ro in range(s):
                for co in range(s):
                    cell=s-1
                    nR=(30-ro-cell)//s + 1
                    nC=(30-co-cell)//s + 1
                    if nR >= 2 and nC >= 2:
                        idx=len(self.configs)
                        self.configs.append((s,ro,co,nR,nC,cell))
                        self.register_buffer(f'Lr_{idx}', _left(nC))
                        self.register_buffer(f'Rr_{idx}', _right(nC))
                        self.register_buffer(f'Lc_{idx}', _left(nR))
                        self.register_buffer(f'Rc_{idx}', _right(nR))
                        self.register_buffer(f'RM_{idx}', _rowmap(s,ro,nR))
                        self.register_buffer(f'CMt_{idx}', _rowmap(s,co,nC).T)

    def _candidate(self,x,idx,s,ro,co,nR,nC,cell):
        acc=None
        for dr in range(cell):
            for dc in range(cell):
                part=x[:,1:,ro+dr:ro+dr+s*nR:s, co+dc:co+dc+s*nC:s]
                acc=part if acc is None else acc+part
        block=(acc > (cell*cell - 0.5)).to(x.dtype)

        left=torch.matmul(block, getattr(self,f'Lr_{idx}'))
        right=torch.matmul(block, getattr(self,f'Rr_{idx}'))
        horiz=((left>0.5)&(right>0.5)).to(x.dtype)

        bt=block.transpose(-2,-1)
        up=torch.matmul(bt, getattr(self,f'Lc_{idx}')).transpose(-2,-1)
        down=torch.matmul(bt, getattr(self,f'Rc_{idx}')).transpose(-2,-1)
        vert=((up>0.5)&(down>0.5)).to(x.dtype)

        cells=torch.clamp(block+horiz+vert,0.0,1.0)
        pix=torch.matmul(getattr(self,f'RM_{idx}'), cells)
        pix=torch.matmul(pix, getattr(self,f'CMt_{idx}'))
        return pix

    def forward(self,x):
        pix=None
        for idx,cfg in enumerate(self.configs):
            cand=self._candidate(x,idx,*cfg)
            pix=cand if pix is None else pix+cand
        pix=torch.clamp(pix,0.0,1.0)
        empty=x[:,0:1,:,:]
        add=pix*empty
        anyadd=torch.clamp(add.sum(dim=1,keepdim=True),0.0,1.0)
        y0=x[:,0:1,:,:]*(1.0-anyadd)
        yrest=torch.clamp(x[:,1:,:,:]+add,0.0,1.0)
        return torch.cat([y0,yrest],dim=1)


In [9]:

with open(TASK_PATH) as f:
    task=json.load(f)
model=Task009MultiLatticeRelation().eval()


In [10]:

dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
dummy[:,0,:,:]=1.0
# realistic non-empty dummy prevents over-aggressive constant-folding
for color,r,c in [(2,3,3),(2,3,15),(3,12,3),(3,12,9)]:
    dummy[:,0,r:r+2,c:c+2]=0.0
    dummy[:,color,r:r+2,c:c+2]=1.0

torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],
                  opset_version=17,dynamic_axes=None,do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m)
ops=op_counts(m)
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
health={
    'task_id':TASK_ID,
    'model_type':'multi-lattice relation-completion PyTorch module exported to static ONNX',
    'not_lookup': True,
    'uses_tree_based_method': False,
    'exact_input_output_bank': False,
    'input_shape': onnx_shape(m.graph.input[0]),
    'output_shape': onnx_shape(m.graph.output[0]),
    'size_bytes': ONNX_PATH.stat().st_size,
    'op_counts': ops,
    'forbidden_ops_present': sorted(forbidden.intersection(ops)),
    'accuracy': {}
}


/tmp/ipykernel_16/1347950781.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],


In [11]:

sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
for split in ['train','test','arc-gen']:
    res=exact_eval(sess,task.get(split,[]))
    health['accuracy'][split]=f"{res['exact']}/{res['total']}"
    if res['first_wrong'] is not None:
        raise AssertionError((split,res['first_wrong']))

ag=task.get('arc-gen',[])
fit_n=int(len(ag)*0.4)
fit=exact_eval(sess,ag[:fit_n]); hold=exact_eval(sess,ag[fit_n:])
health['accuracy']['arc_gen_40_60']=f"fit {fit['exact']}/{fit['total']}, test {hold['exact']}/{hold['total']}"

assert health['input_shape']==[1,10,30,30]
assert health['output_shape']==[1,10,30,30]
assert health['size_bytes']<1_400_000
assert not health['forbidden_ops_present']

with open(HEALTH_PATH,'w') as f:
    json.dump(health,f,indent=2)
health


{'task_id': 'task009',
 'model_type': 'multi-lattice relation-completion PyTorch module exported to static ONNX',
 'not_lookup': True,
 'uses_tree_based_method': False,
 'exact_input_output_bank': False,
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 291897,
 'op_counts': {'Identity': 266,
  'Constant': 1182,
  'Slice': 206,
  'Add': 680,
  'Greater': 250,
  'Cast': 150,
  'MatMul': 300,
  'And': 100,
  'Transpose': 150,
  'Clip': 53,
  'Mul': 2,
  'ReduceSum': 1,
  'Sub': 1,
  'Concat': 1},
 'forbidden_ops_present': [],
 'accuracy': {'train': '3/3',
  'test': '1/1',
  'arc-gen': '261/261',
  'arc_gen_40_60': 'fit 104/104, test 157/157'}}

In [12]:

# Kaggle submission zip: root contains only task009.onnx
for zp in [ZIP_PATH, GENERIC_ZIP]:
    if zp.exists():
        zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
ZIP_PATH, GENERIC_ZIP


(PosixPath('/kaggle/working/task009_static_graph_submission.zip'),
 PosixPath('/kaggle/working/submission.zip'))